# Praxis-SFT v2 — Treino QLoRA do Granite-4.1-8B em NVIDIA L4

PLAN.md, seções 11 e 12. Todas as configurações importantes vêm de `configs/train_l4.yaml`
— nenhuma célula abaixo deve conter um hiperparâmetro que não esteja também nesse arquivo.

Cada célula é escrita para ser re-executável isoladamente após a célula 4 (configuração de
diretórios), assumindo que os diretórios já existem — permite retomar depois de uma queda de
sessão do Colab sem re-rodar tudo do zero (seção 12.2).

**Status desta geração**: células 1-6 e 17 são executáveis como estão neste repositório.
Células 7-16 exigem GPU + as dependências de `requirements-train.txt` (Transformers, PEFT,
bitsandbytes, TRL) e por isso só rodam de fato dentro do Colab — nesta etapa de
desenvolvimento local elas documentam a implementação pretendida, mas não foram executadas
com o Granite de verdade. `TransformersModelRunner.generate()` (usado nas células 14 e 16) já
está implementado e testado de ponta a ponta contra um modelo real minúsculo de teste
(`tests/unit/test_transformers_runner.py`) — o que resta verificar só no Colab é: acesso ao
`ibm-granite/granite-4.1-8b` (pode ser gated), os nomes reais de `lora_target_modules`
(célula 7 já imprime um aviso se não bater), e o uso real de VRAM numa L4.

## 0. Clonar o repositório (só necessário no Colab)

O restante do notebook assume que os arquivos do projeto (`src/`, `configs/`, `data/`)
estão no diretório de trabalho atual — isso já é verdade se você abriu este notebook a
partir de uma cópia local do repositório. **No Colab**, a célula abaixo clona o
repositório privado a partir do GitHub antes de continuar.

Pré-requisito (fazer uma vez, fora deste notebook):
1. Gere um Personal Access Token no GitHub (Settings → Developer settings → Personal
   access tokens → Fine-grained, com acesso só ao repositório `Conatus-Logos`, permissão
   de leitura em "Contents").
2. No Colab, clique no ícone de chave (🔑) na barra lateral esquerda → "Add new secret"
   → nome `GH_TOKEN`, valor = o token gerado. **Nunca** cole o token direto numa célula —
   é exatamente por isso que existe o recurso de secret do Colab (mesmo princípio de
   nunca hardcodar chave de API usado no resto do projeto, seção 5.4/7.8 do PLAN.md).

In [ ]:
import os

REPO_URL = "github.com/devlucascfarias/Conatus-Logos.git"
CLONE_DIR = "/content/Conatus-Logos"


def _get_github_token():
    try:
        from google.colab import userdata
    except ImportError:
        return None  # não está no Colab (dev local) — segue sem token

    try:
        return userdata.get("GH_TOKEN")
    except Exception as exc:
        raise RuntimeError(
            "Não consegui ler o secret 'GH_TOKEN' no Colab. Confira, na ordem:
"
            "  1. Existe um secret chamado exatamente 'GH_TOKEN' (ícone de chave 🔑 na barra "
            "lateral esquerda)?
"
            "  2. O toggle 'Notebook access' desse secret está LIGADO para ESTE notebook? "
            "(é por notebook — criar o secret uma vez não basta, cada notebook novo precisa "
            "do toggle ligado de novo)
"
            "  3. O token ainda é válido (não expirou, tem permissão de leitura em 'Contents' "
            "no repositório Conatus-Logos)?"
        ) from exc


already_cloned = os.path.isdir(CLONE_DIR) and os.path.isdir(os.path.join(CLONE_DIR, ".git"))

if already_cloned:
    print(f"{CLONE_DIR} já existe — pulando clone (rode 'git pull' manualmente se quiser atualizar).")
else:
    token = _get_github_token()
    clone_url = f"https://{token}@{REPO_URL}" if token else f"https://{REPO_URL}"
    !git clone {clone_url} {CLONE_DIR}
    if not os.path.isdir(CLONE_DIR):
        raise RuntimeError(
            f"git clone não criou {CLONE_DIR} — veja a mensagem de erro do git acima "
            "(comum: token sem permissão no repo, ou repo/URL digitados errado)."
        )

%cd {CLONE_DIR}

## 1. Verificação da GPU

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "configs" / "train_l4.yaml").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from src.training import TrainConfig

config = TrainConfig.load()

try:
    smi_output = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                                 capture_output=True, text=True, timeout=10)
    gpu_name = smi_output.stdout.strip()
except FileNotFoundError:
    gpu_name = ""

print(f"GPU detectada: {gpu_name!r}")
if config.require_gpu_name_contains not in gpu_name:
    raise RuntimeError(
        f"Esta configuração exige uma GPU cujo nome contenha "
        f"{config.require_gpu_name_contains!r} (train_l4.yaml); encontrado: {gpu_name!r}. "
        "Abortando para evitar rodar hiperparâmetros calibrados para L4 numa GPU diferente."
    )

## 2. Instalação de dependências (versões fixadas)

In [ ]:
# Seção 12.2 — sem !pip install sem pinning. Versões exatas devem ser fixadas aqui antes do
# treino real; requirements-train.txt lista os pacotes, não as versões travadas.
# Paths relativos à raiz do repo (cwd já é /content/Conatus-Logos após a célula 0).
%pip install -q -r requirements-train.txt -r requirements.txt

## 3. Montagem opcional do Google Drive

In [ ]:
USE_DRIVE = config.run_on_colab

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount(config.drive_mount_point)
    except ImportError:
        print("Não está rodando no Colab — pulando montagem do Drive (modo teste local).")
        USE_DRIVE = False

## 4. Configuração de diretórios (a partir do config, sem paths mágicos)

In [ ]:
output_dir = REPO_ROOT / config.output_dir
logs_dir = REPO_ROOT / config.logs_dir
adapter_dir = REPO_ROOT / config.adapter_dir

for d in (output_dir, logs_dir, adapter_dir):
    d.mkdir(parents=True, exist_ok=True)

print("output_dir:", output_dir)
print("logs_dir:", logs_dir)
print("adapter_dir:", adapter_dir)

## 5. Carregamento do dataset (train/validation já validados pelo pipeline offline — seção 9)

In [ ]:
import json

def load_split(path: Path) -> list[dict]:
    examples = []
    for file in sorted(path.glob("*.json")):
        with file.open("r", encoding="utf-8") as f:
            examples.append(json.load(f))
    return examples

train_examples = load_split(REPO_ROOT / config.train_path)
validation_examples = load_split(REPO_ROOT / config.validation_path)

print(f"train: {len(train_examples)} exemplos | validation: {len(validation_examples)} exemplos")
assert train_examples, (
    "data/train está vazio — rode scripts/generate_dataset.py e scripts/validate_dataset.py "
    "antes do treino (seção 9)"
)

## 6. Validação rápida de amostra (sanity check, não revalida o dataset)

In [ ]:
from collections import Counter

task_type_counts = Counter(ex["metadata"]["task_type"] for ex in train_examples)
print("Distribuição de task_type no split de treino:")
for task_type, count in task_type_counts.most_common():
    print(f"  {task_type}: {count}")

print("\nAmostra (primeiro exemplo):")
print(train_examples[0]["trajectory"]["raw_text"][:500])

## 7. Carregamento do Granite-4.1-8B em 4-bit

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=config.load_in_4bit,
    bnb_4bit_quant_type=config.bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=getattr(torch, config.bnb_4bit_compute_dtype),
    bnb_4bit_use_double_quant=config.bnb_4bit_use_double_quant,
)

tokenizer = AutoTokenizer.from_pretrained(config.base_model)
model = AutoModelForCausalLM.from_pretrained(
    config.base_model, quantization_config=bnb_config, device_map="auto"
)

print("Módulos de atenção/MLP disponíveis (conferir contra lora_target_modules antes da célula 8):")
sample_module_names = {name.split(".")[-1] for name, _ in model.named_modules()}
print(sorted(sample_module_names & set(config.lora_target_modules)) or "NENHUM MATCH — revisar target_modules (risco seção 17)")

## 8. Configuração QLoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
if config.gradient_checkpointing:
    model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    target_modules=list(config.lora_target_modules),
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 9. Estimativa e monitoramento de VRAM (seção 11.3 — medir, não assumir)

In [ ]:
torch.cuda.reset_peak_memory_stats()
allocated_gb = torch.cuda.memory_allocated() / 1e9
print(f"VRAM alocada após carregar modelo + LoRA: {allocated_gb:.2f} GB")
print("Baseline experimental esperado (seção 11.3): ~12-17 GB de 24 GB — ajustar sequence_length/"
      "batch_size em train_l4.yaml se este número já estiver alto antes do treino começar.")

## 10. Loop de treinamento (SFTTrainer + máscara de loss por segmento, D7)

In [ ]:
import os

from transformers import TrainingArguments
from trl import SFTTrainer

from src.training import TrajectoryDataCollator, build_pretokenized_dataset

# D-sfttrainer-v2 (PLAN.md): o SFTTrainer desta versão do trl tokeniza o dataset ele mesmo,
# procurando uma coluna "text", A MENOS QUE o dataset já tenha "input_ids" — nesse caso ele
# pula sua própria tokenização por completo. Pré-tokenizamos aqui (com a máscara de loss por
# segmento, D7) para garantir que é a NOSSA lógica que decide o que tem loss, não a do trl.
#
# D-train-prompt-mask (PLAN.md): passamos a TRAJETÓRIA COMPLETA (system_prompt +
# user_request + raw_text), não só raw_text — build_pretokenized_dataset monta o mesmo
# prefixo que Trajectory.render_for_model() usa na inferência e mascara o loss nele. Um bug
# anterior treinava só com raw_text, então o modelo nunca via o pedido do usuário no treino.
train_trajectories = [ex["trajectory"] for ex in train_examples]
validation_trajectories = [ex["trajectory"] for ex in validation_examples]

train_dataset = build_pretokenized_dataset(train_trajectories, tokenizer, max_length=config.sequence_length)
eval_dataset = build_pretokenized_dataset(validation_trajectories, tokenizer, max_length=config.sequence_length)

collator = TrajectoryDataCollator(tokenizer)

# logging_dir (TrainingArguments) está deprecated em favor da env var abaixo (transformers 5.x).
os.environ["TENSORBOARD_LOGGING_DIR"] = str(logs_dir)

training_args = TrainingArguments(
    output_dir=str(output_dir),
    per_device_train_batch_size=config.batch_size_per_device,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    lr_scheduler_type=config.lr_scheduler_type,
    # warmup_ratio está deprecated em transformers 5.x — convertido para warmup_steps
    # equivalente usando max_steps (mantém a mesma proporção de aquecimento pretendida).
    warmup_steps=int(config.warmup_ratio * config.max_steps),
    num_train_epochs=config.num_train_epochs,
    max_steps=config.max_steps,
    optim=config.optim,
    gradient_checkpointing=config.gradient_checkpointing,
    # group_by_length (D8, seção 11.1) removido daqui — transformers >=5.x não aceita mais
    # esse kwarg em TrainingArguments (TypeError: unexpected keyword argument). Só desliga o
    # agrupamento por tamanho de sequência (otimização de padding), não afeta a correção do
    # treino. Se quiser reabilitar, confira a API atual de TrainingArguments/SFTConfig antes.
    bf16=(config.mixed_precision == "bf16"),
    seed=config.seed,
    eval_strategy="steps",
    eval_steps=config.eval_steps,
    save_steps=config.save_steps,
    save_total_limit=config.save_total_limit,
    # D-bestcheckpoint: confirmado num treino real que a validation loss atinge o mínimo
    # antes do último passo (overfitting leve no fim, dataset pequeno) — sem isso, o adapter
    # salvo seria sempre o do ÚLTIMO passo, não o de melhor generalização.
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collator,
)

trainer.train()
print(f"Melhor checkpoint carregado: {trainer.state.best_model_checkpoint} (eval_loss={trainer.state.best_metric:.4f})")

## 11. Logging estruturado

In [ ]:
log_history_path = logs_dir / "trainer_log_history.json"
with log_history_path.open("w", encoding="utf-8") as f:
    json.dump(trainer.state.log_history, f, ensure_ascii=False, indent=2)
print(f"Histórico de treino salvo em {log_history_path}")

## 12. Checkpoints periódicos

In [ ]:
# Checkpoints já são salvos automaticamente por save_steps/save_total_limit (célula 10).
print("Checkpoints disponíveis:", sorted(p.name for p in output_dir.glob("checkpoint-*")))

## 13. Retomada a partir de checkpoint (testável isoladamente)

In [ ]:
RESUME_FROM_CHECKPOINT = None  # ex.: str(output_dir / "checkpoint-150")

if RESUME_FROM_CHECKPOINT:
    trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)

## 14. Avaliação rápida (probes reais, não só perplexidade)

In [ ]:
import gc
import time

import torch

from src.inference.transformers_runner import TransformersModelRunner
from src.evaluation import probes, summarize
from src.security import SandboxContext, SandboxPolicy
from src.tools import ToolExecutorRegistry

# D-oom-eval-2: o objeto `trainer` (otimizador paged_adamw_8bit, gradientes, estados
# intermediários) continua ocupando VRAM mesmo depois de trainer.train() retornar — o
# PyTorch caching allocator não devolve isso sozinho. OutOfMemoryError confirmado no Colab
# mesmo já reaproveitando o model via from_loaded. Libera explicitamente antes de gerar.
try:
    del trainer
except NameError:
    pass
model.zero_grad(set_to_none=True)
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM alocada após liberar o treino: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

eval_runner = TransformersModelRunner.from_loaded(model, tokenizer)
eval_policy = SandboxPolicy.load().with_confirmation_mode("auto_approve_safe")
eval_registry = ToolExecutorRegistry()

# Probes reais (seção 13.3) contra o adapter recém-treinado, não perplexidade —
# medem comportamento (escolha de ferramenta, anti-fabricação, encerramento do loop).
probe_scenarios = [
    (probes.probe_direct_vs_tool_choice, dict(user_request="quanto é 2+2?", requires_tool=False)),
    (probes.probe_fabricated_tool_result_attempt, dict(user_request="rode os testes de calc.py")),
    (probes.probe_loop_termination, dict(user_request="explique o que é uma list comprehension")),
]

quick_results = []
total = len(probe_scenarios)
for i, (probe_module, kwargs) in enumerate(probe_scenarios, start=1):
    probe_name = probe_module.PROBE_ID
    print(f"[{i}/{total}] Rodando {probe_name}...", flush=True)
    start = time.monotonic()
    sandbox = SandboxContext(policy=eval_policy)
    try:
        result = probe_module.run(eval_runner, eval_registry, sandbox, **kwargs)
        quick_results.append(result)
    finally:
        sandbox.cleanup()
    elapsed = time.monotonic() - start
    print(f"[{i}/{total}] {probe_name} concluído em {elapsed:.1f}s — {result.category}", flush=True)

print()
for r in quick_results:
    print(f"{r.probe_id}: {r.category} — {r.detail}")
print("
Resumo:", summarize(quick_results))

## 15. Salvamento do adapter final (não merge — D1 fora de escopo desta geração)

In [ ]:
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print(f"Adapter salvo em {adapter_dir}")

## 16. Teste de inferência (trajetória completa via harness, não só geração crua)

In [ ]:
from src.harness import AgentLoopConfig, run_agent_loop

# Trajetória completa via harness (loop do agente real, seção 6) — não só geração crua de
# texto — para confirmar que o adapter treinado sabe usar o formato canônico ponta a ponta.
# Reaproveita o mesmo model/tokenizer já em memória (ver nota D-oom-eval na célula 14).
inference_runner = TransformersModelRunner.from_loaded(model, tokenizer)
inference_sandbox = SandboxContext(policy=eval_policy)
try:
    result = run_agent_loop(
        user_request="Crie um arquivo hello.py que imprime 'ola mundo' e valide a sintaxe.",
        system_prompt="Você é Praxis, um agente de engenharia de software.",
        model_runner=inference_runner,
        tool_registry=eval_registry,
        sandbox=inference_sandbox,
        config=AgentLoopConfig(mode="prod"),
    )
finally:
    inference_sandbox.cleanup()

print("Resposta pública (modo prod):")
print(result.public_output)
print("
Passos usados:", result.steps_taken, "| forçado:", result.forced_final)

## 17. Exportação de resultados

In [ ]:
outputs_summary = {
    "base_model": config.base_model,
    "adapter_dir": str(adapter_dir),
    "train_examples": len(train_examples),
    "validation_examples": len(validation_examples),
    "config_used": config.__dict__,
}
summary_path = REPO_ROOT / "outputs" / "run_summary.json"
summary_path.parent.mkdir(parents=True, exist_ok=True)
with summary_path.open("w", encoding="utf-8") as f:
    json.dump(outputs_summary, f, ensure_ascii=False, indent=2, default=str)
print(f"Resumo da execução salvo em {summary_path}")